In [247]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE

df = pd.read_csv('creditcard.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'creditcard.csv'

In [ ]:
df.head()

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

scaler = StandardScaler()
X[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=42
)

pca = PCA(n_components=0.80)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)

In [ ]:
rf_preds = rf.predict(X_test)
xgb_preds = xgb.predict(X_test)

rf_acc = accuracy_score(y_test, rf_preds)
xgb_acc = accuracy_score(y_test, xgb_preds)

rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])

print(f"Random Forest -> Accuracy: {rf_acc:.4f}, ROC-AUC: {rf_auc:.4f}")
print(f"XGBoost       -> Accuracy: {xgb_acc:.4f}, ROC-AUC: {xgb_auc:.4f}")

In [ ]:
rf_importance = rf.feature_importances_
rf_indices = np.argsort(rf_importance)[-10:]

plt.figure(figsize=(8, 5))
plt.barh(range(len(rf_indices)), rf_importance[rf_indices], align='center')
plt.yticks(range(len(rf_indices)), [X.columns[i] for i in rf_indices])
plt.title("Random Forest Feature Importances")
plt.show()

In [ ]:
xgb_importance = xgb.feature_importances_
xgb_indices = np.argsort(xgb_importance)[-10:]

plt.figure(figsize=(8, 5))
plt.barh(range(len(xgb_indices)), xgb_importance[xgb_indices], align='center', color='green')
plt.yticks(range(len(xgb_indices)), [X.columns[i] for i in xgb_indices])
plt.title("XGBoost Feature Importances")
plt.show()